# Remote Reading with cytozip

cytozip supports reading `.cz` files directly from HTTP/HTTPS URLs without downloading the entire file.
This is achieved through:
- **HTTP Range requests** — only the required bytes are fetched from the server
- **Chunk index** — enables O(1) lookup of chunk positions (no sequential scanning)
- **Read-ahead cache** — minimizes the number of HTTP requests via a 2MB cache

Requirements:
- The `.cz` file must include a chunk index (written by default)
- The HTTP server must support Range requests (most modern servers do)

## 1. Installation

```shell
pip install ChunkZIP
# or
pip install git+https://github.com/DingWB/cytozip.git
```

## 2. Opening a remote .cz file

There are two ways to open a remote `.cz` file:

**Method 1: `Reader.from_url()` (recommended)**
```python
reader = cytozip.Reader.from_url(url, cache_size=2*1024*1024)
```

**Method 2: Pass URL directly to `Reader()` (auto-detected)**
```python
reader = cytozip.Reader(url)
```

Both methods detect `http://` or `https://` prefixes and automatically use the `RemoteFile` backend.

In [1]:
import cytozip
from cytozip.cz import RemoteFile
print(f"cytozip version: {cytozip.__version__}")

2026-04-05 23:45:56.648 | DEBUG    | cytozip.cz:<module>:84 - Cython accelerated functions loaded successfully


cytozip version: 0.1.0.dev4


### Example: Open and inspect a remote file

Replace `url` below with the URL of your `.cz` file:

In [2]:
# Uncomment and replace with your URL:
# url = "https://your-server.com/data/sample.cz"
# reader = cytozip.Reader.from_url(url)
# reader.print_header()
print("Replace the URL above with a real .cz file URL to test remote reading.")

Replace the URL above with a real .cz file URL to test remote reading.


## 3. Querying remote files

Once opened, all Reader methods work identically on remote and local files:

In [3]:
# reader = cytozip.Reader.from_url(url)
#
# # Summary of all chunks
# reader.summary_chunks()
#
# # Fetch all records for a chunk
# for record in reader.fetch(("chr1",)):
#     print(record)
#
# # Query by coordinate range
# for record in reader.query(Dimension="chr1", start=3000000, end=3001000, printout=False):
#     print(record)
#
# # Fetch raw bytes for numpy processing
# import numpy as np
# raw_bytes = reader.fetch_chunk_bytes(("chr1",))
# # Parse with np.frombuffer based on your format
#
# reader.close()
print("All Reader methods (fetch, query, summary_chunks, etc.) work on remote files.")

All Reader methods (fetch, query, summary_chunks, etc.) work on remote files.


## 4. Chunk Index

The chunk index (written at the end of .cz files) maps each dimension tuple to its file offset.
This enables O(1) chunk lookup without scanning the entire file — critical for remote access.

In [4]:
# Read chunk index from a local .cz file
import struct, tempfile, os

# Create a .cz file with chunk index
tmpfile = tempfile.mktemp(suffix=".cz")
w = cytozip.Writer(
    Output=tmpfile,
    Formats=["Q", "H", "H"],
    Columns=["pos", "mc", "cov"],
    Dimensions=["chrom"],
    delta_cols=[0],
)
for chrom in ["chr1", "chr2", "chr3"]:
    records = [(100, 1, 10), (200, 2, 15), (350, 3, 20)]
    data = b"".join(struct.pack("<QHH", *r) for r in records)
    w.write_chunk(data, [chrom])
w.close()

# Read chunk index
r = cytozip.Reader(tmpfile)
idx = r.read_chunk_index()
if idx:
    print("Chunk index entries:")
    for dim, offset in idx.items():
        print(f"  {dim} -> offset {offset}")
r.close()
os.unlink(tmpfile)

TypeError: Writer.__init__() got an unexpected keyword argument 'delta_cols'

## 5. RemoteFile (Low-Level API)

`RemoteFile` is a file-like object that reads from HTTP URLs using Range requests.
It implements `read()`, `seek()`, `tell()`, and `close()`, making it a drop-in replacement for `open(path, 'rb')`.

Features:
- **Read-ahead cache**: Pre-fetches data in configurable chunks (default 2MB) to minimize HTTP requests
- **Seek support**: `SEEK_SET`, `SEEK_CUR`, and `SEEK_END` are all supported
- **Size property**: Total file size from Content-Length header

In [5]:
from cytozip.cz import RemoteFile

# Usage:
# rf = RemoteFile("https://your-server.com/data/file.cz", cache_size=2*1024*1024)
# print(f"File size: {rf.size} bytes")
#
# # Read first 100 bytes
# header = rf.read(100)
#
# # Seek to a specific position
# rf.seek(1000)
# data = rf.read(50)
#
# # Seek from end (e.g., read last 100 bytes for chunk index)
# rf.seek(-100, 2)
# tail = rf.read(100)
#
# rf.close()

print("RemoteFile API:")
print("  RemoteFile(url, cache_size=2*1024*1024)")
print("  .read(size)     - read bytes with read-ahead caching")
print("  .seek(off, wh)  - seek to position (SEEK_SET/CUR/END)")
print("  .tell()         - return current file position")
print("  .size           - total file size (Content-Length)")
print("  .close()        - close the connection")

RemoteFile API:
  RemoteFile(url, cache_size=2*1024*1024)
  .read(size)     - read bytes with read-ahead caching
  .seek(off, wh)  - seek to position (SEEK_SET/CUR/END)
  .tell()         - return current file position
  .size           - total file size (Content-Length)
  .close()        - close the connection


## 6. Local Demo with HTTP Server

For testing, you can serve local `.cz` files via Python's built-in HTTP server:

```shell
# In one terminal: serve the current directory
cd /path/to/cz/files
python -m http.server 8765

# In another terminal or Python session:
python -c "
import cytozip
r = cytozip.Reader.from_url('http://localhost:8765/sample.cz')
r.print_header()
for record in r.fetch(('chr1',)):
    print(record)
r.close()
"
```

In [6]:
# Full local demo (creates a temp file, serves it, reads remotely)
import struct, tempfile, os, threading, time
from http.server import HTTPServer, SimpleHTTPRequestHandler

# 1. Create a .cz file
tmpdir = tempfile.mkdtemp()
czfile = os.path.join(tmpdir, "demo.cz")
w = cytozip.Writer(
    Output=czfile,
    Formats=["Q", "H", "H"],
    Columns=["pos", "mc", "cov"],
    Dimensions=["chrom"],
)
data = b"".join(struct.pack("<QHH", pos, mc, cov)
                for pos, mc, cov in [(100, 1, 10), (200, 2, 15), (350, 3, 20)])
w.write_chunk(data, ["chr1"])
data = b"".join(struct.pack("<QHH", pos, mc, cov)
                for pos, mc, cov in [(50, 0, 5), (150, 1, 8)])
w.write_chunk(data, ["chr2"])
w.close()
print(f"Created: {czfile} ({os.path.getsize(czfile)} bytes)")

# 2. Start HTTP server in background
os.chdir(tmpdir)
handler = SimpleHTTPRequestHandler
handler.log_message = lambda *args: None  # suppress logs
server = HTTPServer(("", 0), handler)
port = server.server_address[1]
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print(f"Serving on port {port}")

# 3. Read remotely
url = f"http://localhost:{port}/demo.cz"
reader = cytozip.Reader.from_url(url)
print(f"\nRemote is_remote: {reader._is_remote}")
print(f"Chunks: {list(reader.dim2chunk_start.keys())}")

print("\n--- chr1 records (remote) ---")
for record in reader.fetch(("chr1",)):
    print(record)

print("\n--- chr2 records (remote) ---")
for record in reader.fetch(("chr2",)):
    print(record)

reader.close()
server.shutdown()

# Cleanup
os.unlink(czfile)
os.rmdir(tmpdir)
print("\nDemo complete!")

Created: /tmp/tmpx6hhtc6_/demo.cz (310 bytes)
Serving on port 42935

Remote is_remote: True
Chunks: [('chr1',), ('chr2',)]

--- chr1 records (remote) ---
[100, 1, 10]
[200, 2, 15]
[350, 3, 20]
[100, 1, 10]
[200, 2, 15]
[350, 3, 20]

--- chr2 records (remote) ---
[50, 0, 5]
[150, 1, 8]
[50, 0, 5]
[150, 1, 8]



Demo complete!


## 7. Performance Tips

- **Cache size**: Increase `cache_size` for sequential reads over large chunks:
  ```python
  reader = cytozip.Reader.from_url(url, cache_size=10*1024*1024)  # 10MB cache
  ```
- **Chunk index**: .cz files include a chunk index by default. Without it, the reader must scan the entire file sequentially.
- **Query vs fetch**: For large files, prefer `query()` with coordinate ranges over `fetch()` on the entire chunk — this minimizes data transferred.
- **Server requirements**: The HTTP server must support:
  - `Range` request header
  - `Content-Length` response header
  - `206 Partial Content` response status